In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
import pandas as pd
from sklearn.cluster import HDBSCAN
import umap
from tqdm import tqdm

In [ ]:
asserts = pd.read_csv(
    "data/shome2023notebook/asserts.csv",
    header=None,
    names=["notebook", "stmt"],
)
asserts

In [ ]:
lasts = pd.read_csv(
    "data/shome2023notebook/lasts.csv",
    header=None,
    names=["notebook", "stmt"],
)
lasts

In [ ]:
data = pd.concat([asserts, lasts])
data.loc[data["notebook"].str.contains(r"^data/assert"), "source"] = "GH"
data.loc[data["notebook"].str.contains(r"^data/quaranta"), "source"] = "KG"
data = data.sample(frac=0.1)
data = data.dropna()
data

In [ ]:
MODEL = "microsoft/codebert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModel.from_pretrained(MODEL)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
def embed_batch(texts: list[str], batch_size: int = 32) -> np.ndarray:
    """Return CLS-token embeddings, shape (n, 768)."""
    all_vecs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
        batch = texts[i : i + batch_size]
        enc = tokenizer (
            batch,
            padding=True,
            truncation=True,
            max_length=128, # assert statements are short
            return_tensors="pt",
        ).to(device)
        with torch.no_grad():
            out = model(**enc)
        # CLS token = out.last_hidden_state[:, 0, :]
        vecs = out.last_hidden_state[:, 0, :].cpu().numpy()
        all_vecs.append(vecs)
    return np.vstack(all_vecs)

In [ ]:
X_ALL = embed_batch(data.loc[:, "stmt"].to_list()) # (n_samples, 768)
# X_GH = embed_batch(data.loc[data["source"] == "GH", "stmt"].to_list()) # (n_samples, 768)
# X_KG = embed_batch(data.loc[data["source"] == "KG", "stmt"].to_list()) # (n_samples, 768)

In [ ]:
X_ALL.shape

In [ ]:
np.save("data/shome2023notebook/X_ALL_01", X_ALL)

In [ ]:
X_ALL = np.load("data/shome2023notebook/X_ALL_01.npy")
X_ALL.shape

# Experimentations with Clustering

Experimenting with different UMAP and HDBSCAN configuration to reduce the number of total clusters.

## Baseline

In [ ]:
umap_params = dict(
    n_components=10,
    n_neighbors=15,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
)

hdbscan_params = dict(
    min_cluster_size=10,
    min_samples=5,
    metric="euclidean",
    cluster_selection_method="eom",
)

reducer = umap.UMAP(**umap_params)
clusterer = HDBSCAN(**hdbscan_params)

In [ ]:
X_ALL_reduced = reducer.fit_transform(X_ALL)
print(f"X_ALL_reduced shape: {X_ALL_reduced.shape}")
labels = clusterer.fit_predict(X_ALL_reduced)
data.loc[:, "CALL"] = labels

# X_GH = reducer.fit_transform(X_GH)
# labels = clusterer.fit_predict(X_GH)
# data.loc[data["source"] == "GH", "CGH"] = labels

# X_KG = reducer.fit_transform(X_KG)
# labels = clusterer.fit_predict(X_KG)
# data.loc[data["source"] == "KG", "CKG"] = labels
# data

In [ ]:
call = data.groupby("CALL")["stmt"].count()
call.describe()

## EXP 1

Increased HDBSCAN `min_cluster_size` to 25.

In [ ]:
umap_params = dict(
    n_components=10,
    n_neighbors=15,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
)

hdbscan_params = dict(
    min_cluster_size=25,
    min_samples=5,
    metric="euclidean",
    cluster_selection_method="eom",
)

reducer = umap.UMAP(**umap_params)
clusterer = HDBSCAN(**hdbscan_params)

In [ ]:
X_ALL_reduced = reducer.fit_transform(X_ALL)
print(f"X_ALL_reduced shape: {X_ALL_reduced.shape}")
labels = clusterer.fit_predict(X_ALL_reduced)
data.loc[:, "CALL"] = labels

# X_GH = reducer.fit_transform(X_GH)
# labels = clusterer.fit_predict(X_GH)
# data.loc[data["source"] == "GH", "CGH"] = labels

# X_KG = reducer.fit_transform(X_KG)
# labels = clusterer.fit_predict(X_KG)
# data.loc[data["source"] == "KG", "CKG"] = labels
# data

In [ ]:
call = data.groupby("CALL")["stmt"].count()
call.describe()

Cluster size went down to 128 (60% reduction compared to baseline).

## EXP 2

- Increased UMAP `n_neighbors` to 30.

In [ ]:
umap_params = dict(
    n_components=10,
    n_neighbors=30,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
)

hdbscan_params = dict(
    min_cluster_size=10,
    min_samples=5,
    metric="euclidean",
    cluster_selection_method="eom",
)

reducer = umap.UMAP(**umap_params)
clusterer = HDBSCAN(**hdbscan_params)

In [ ]:
X_ALL_reduced = reducer.fit_transform(X_ALL)
print(f"X_ALL_reduced shape: {X_ALL_reduced.shape}")
labels = clusterer.fit_predict(X_ALL_reduced)
data.loc[:, "CALL"] = labels

# X_GH = reducer.fit_transform(X_GH)
# labels = clusterer.fit_predict(X_GH)
# data.loc[data["source"] == "GH", "CGH"] = labels

# X_KG = reducer.fit_transform(X_KG)
# labels = clusterer.fit_predict(X_KG)
# data.loc[data["source"] == "KG", "CKG"] = labels
# data

In [ ]:
call = data.groupby("CALL")["stmt"].count()
call.describe()

Cluster size went down to 276 (15% reduction compared to baseline).

## EXP 3

- Increased UMAP `n_neighbors` to 30.
- Decreased UMAP `n_components` to 5.

In [ ]:
umap_params = dict(
    n_components=5,
    n_neighbors=30,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
)

hdbscan_params = dict(
    min_cluster_size=10,
    min_samples=5,
    metric="euclidean",
    cluster_selection_method="eom",
)

reducer = umap.UMAP(**umap_params)
clusterer = HDBSCAN(**hdbscan_params)

In [ ]:
X_ALL_reduced = reducer.fit_transform(X_ALL)
print(f"X_ALL_reduced shape: {X_ALL_reduced.shape}")
labels = clusterer.fit_predict(X_ALL_reduced)
data.loc[:, "CALL"] = labels

# X_GH = reducer.fit_transform(X_GH)
# labels = clusterer.fit_predict(X_GH)
# data.loc[data["source"] == "GH", "CGH"] = labels

# X_KG = reducer.fit_transform(X_KG)
# labels = clusterer.fit_predict(X_KG)
# data.loc[data["source"] == "KG", "CKG"] = labels
# data

In [ ]:
call = data.groupby("CALL")["stmt"].count()
call.describe()

Cluster size went down to 273 (16% reduction compared to baseline).

## EXP 4

Using the default UMAP parameters.

In [ ]:
umap_params = dict(
    random_state=42,
)

hdbscan_params = dict(
    min_cluster_size=10,
    min_samples=5,
    metric="euclidean",
    cluster_selection_method="eom",
)

reducer = umap.UMAP(**umap_params)
clusterer = HDBSCAN(**hdbscan_params)

In [ ]:
X_ALL_reduced = reducer.fit_transform(X_ALL)
print(f"X_ALL_reduced shape: {X_ALL_reduced.shape}")
labels = clusterer.fit_predict(X_ALL_reduced)
data.loc[:, "CALL"] = labels

# X_GH = reducer.fit_transform(X_GH)
# labels = clusterer.fit_predict(X_GH)
# data.loc[data["source"] == "GH", "CGH"] = labels

# X_KG = reducer.fit_transform(X_KG)
# labels = clusterer.fit_predict(X_KG)
# data.loc[data["source"] == "KG", "CKG"] = labels
# data

In [ ]:
call = data.groupby("CALL")["stmt"].count()
call.describe()

Cluster size same as baseline.

## EXP 5

- Increase HDBSCAN `min_cluster_size` to 50.

In [ ]:
umap_params = dict(
    n_components=10,
    n_neighbors=15,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
)

hdbscan_params = dict(
    min_cluster_size=50,
    min_samples=5,
    metric="euclidean",
    cluster_selection_method="eom",
)

reducer = umap.UMAP(**umap_params)
clusterer = HDBSCAN(**hdbscan_params)

In [ ]:
X_ALL_reduced = reducer.fit_transform(X_ALL)
print(f"X_ALL_reduced shape: {X_ALL_reduced.shape}")
labels = clusterer.fit_predict(X_ALL_reduced)
data.loc[:, "CALL"] = labels

# X_GH = reducer.fit_transform(X_GH)
# labels = clusterer.fit_predict(X_GH)
# data.loc[data["source"] == "GH", "CGH"] = labels

# X_KG = reducer.fit_transform(X_KG)
# labels = clusterer.fit_predict(X_KG)
# data.loc[data["source"] == "KG", "CKG"] = labels
# data

In [ ]:
call = data.groupby("CALL")["stmt"].count()
call.describe()

Cluster size went down to 128 (60% reduction compared to baseline).

In [ ]:
data.shape

# Sampling

Proportional stratified sampling.

In [ ]:
call
min, max = call.quantile([0.25, 0.75])
print(f"min: {min}, max: {max}")
call.describe()

In [ ]:
# Reassign clusters outside [Q1, Q3] to a new "other" cluster (label: -2)
out_of_range = cluster_sizes[(cluster_sizes < min) | (cluster_sizes > max)].index
data["CC_reduced"] = data["CC"].where(~data["CC"].isin(out_of_range), other=-2)

print(f"Clusters before: {data['CC'].nunique()}")
print(f"Clusters after:  {data['CC_reduced'].nunique()}  ({len(out_of_range)} merged into -2)")
data["CC_reduced"].value_counts().describe()

Clusters before: 326
Clusters after:  174  (153 merged into -2)


count     174.000000
mean       62.856322
std       543.679763
min        15.000000
25%        17.000000
50%        21.000000
75%        25.000000
max      7193.000000
Name: count, dtype: float64

In [ ]:
SAMPLE_SIZE = 385
round(SAMPLE_SIZE * 1 / len(statements))

In [ ]:
TOTAL_SIZE = 385
sample = (
    statements
    .groupby("cluster")
    .apply(lambda x: x.sample(
        n=max(1, round(TOTAL_SIZE * len(x) / len(statements))),
        random_state=42 
    ))
    .reset_index(level=0)
)

In [ ]:
sample

In [ ]:
sample["cluster"].value_counts().sort_index()